# Pandas 实战案例教学（第5～8晚）

使用 AKShare 获取股票数据，结合 Pandas 进行数据清洗、计算与可视化演示。

## 第5晚：股价趋势与涨跌幅计算

In [ ]:
import akshare as ak
import pandas as pd
import matplotlib.pyplot as plt

# 获取数据
df = ak.stock_zh_a_hist(symbol="600519", period="daily", start_date="20240101", end_date="20240601")
df['日期'] = pd.to_datetime(df['日期'])
df.set_index('日期', inplace=True)

# 计算涨跌额与涨跌幅
df['昨收'] = df['收盘'].shift(1)
df['涨跌额'] = df['收盘'] - df['昨收']
df['涨跌幅'] = df['涨跌额'] / df['昨收'] * 100

# 可视化
plt.figure(figsize=(10,5))
plt.plot(df.index, df['收盘'], label='收盘价')
plt.title("收盘价趋势图")
plt.xlabel("日期")
plt.ylabel("元")
plt.grid(True)
plt.legend()
plt.show()


## 第6晚：筛选高波动交易日 + 多股票对比分析

In [ ]:
# 筛选涨跌幅绝对值大于 3%
high_vol_df = df[df['涨跌幅'].abs() > 3]
high_vol_df.to_excel("高波动交易日.xlsx")

# 多股票对比
symbols = {"600519": "茅台", "000001": "平安银行", "002594": "比亚迪"}
result_df = pd.DataFrame()

for code, name in symbols.items():
    temp = ak.stock_zh_a_hist(symbol=code, period="daily", start_date="20240401", end_date="20240601")
    temp['日期'] = pd.to_datetime(temp['日期'])
    temp.set_index('日期', inplace=True)
    result_df[name] = temp['收盘']

result_df.plot(figsize=(12,6), title="多只股票收盘价对比")
plt.ylabel("收盘价")
plt.grid(True)
plt.show()


## 第7晚：模拟买入收益 + 导出报告

In [ ]:
buy_day = "2024-04-02"
sell_day = "2024-05-31"
buy_price = df.loc[buy_day, '收盘']
sell_price = df.loc[sell_day, '收盘']
profit = (sell_price - buy_price) / buy_price * 100
print(f"收益率：{profit:.2f}%")

# 多只股票模拟收益
buy_sell_df = pd.DataFrame(columns=['股票', '买入价', '卖出价', '收益率'])

for code, name in symbols.items():
    temp = ak.stock_zh_a_hist(symbol=code, period="daily", start_date="20240401", end_date="20240601")
    temp['日期'] = pd.to_datetime(temp['日期'])
    temp.set_index('日期', inplace=True)
    try:
        bp = temp.loc[buy_day, '收盘']
        sp = temp.loc[sell_day, '收盘']
        rate = (sp - bp) / bp * 100
        buy_sell_df.loc[len(buy_sell_df)] = [name, bp, sp, rate]
    except:
        continue

buy_sell_df.to_excel("模拟买入收益率.xlsx", index=False)
buy_sell_df


## 第8晚：自定义振幅指标 + 图文报告输出

In [ ]:
df['振幅'] = (df['最高'] - df['最低']) / df['昨收'] * 100
top_vol = df.sort_values(by='振幅', ascending=False).head(10)
top_vol.to_excel("振幅Top10.xlsx")

plt.figure(figsize=(10,5))
plt.bar(top_vol.index.strftime('%Y-%m-%d'), top_vol['振幅'])
plt.xticks(rotation=45)
plt.title("振幅最大的10个交易日")
plt.ylabel("振幅（%）")
plt.tight_layout()
plt.show()
